# ID Forensics — Colab Workbench

Run only the sections you need. Each section is independent.

**Persistent on Drive** (`My Drive/id-forensics/`): images, weights, eval reports

**Colab Secrets** (🔑 sidebar): `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, `AWS_SESSION_TOKEN`

## Config

In [ ]:
GITHUB_TOKEN = ""   # only if repo is private

# --- Data pipeline toggles ---
SYNC_IMAGES = False    # True: download new images S3→Drive (first run / new labels)
REBUILD_DATASET = True # True: convert labels → YOLO splits (after label changes)

# --- Training ---
TRAIN_STAGE = "stage2"  # "stage1", "stage2", "stage4", or None
RUN_EVAL = True
EVAL_SPLIT = "val"

---
## 1. Connect — Drive + GitHub + deps

Mounts Drive, pulls latest code + labels from GitHub, installs Python packages.
Does **not** touch images or labels format. Run every session.

In [ ]:
import os, sys, importlib, subprocess
from pathlib import Path

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

REPO = Path("/content/id-forensics-model")
user, repo = "ansisvaisla", "id-forensics-model"
url = (
    f"https://{GITHUB_TOKEN}@github.com/{user}/{repo}.git"
    if GITHUB_TOKEN else f"https://github.com/{user}/{repo}.git"
)

os.chdir("/content")
if not REPO.is_dir():
    subprocess.run(["git", "clone", url, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "origin", "main"], check=False)

sys.path.insert(0, str(REPO / "scripts"))
import colab_bootstrap as cb
importlib.reload(cb)

cb.connect(github_token=GITHUB_TOKEN)

---
## 2. Sync images from S3 → Drive

Downloads labeled images to Drive. Skips files already there.

**When to run:** first time, or after adding new labeled images.
**Skip if:** `SYNC_IMAGES = False` and images already on Drive.

In [ ]:
if SYNC_IMAGES:
    cb.sync_images_from_s3()
else:
    n = len(list(cb.DRIVE_RAW_DIR.glob("*.jpg")))
    print(f"Skipped S3 sync — {n} images already on Drive")

---
## 3. Build training datasets

Converts `label_studio_export.json` → YOLO format, creates train/val/test splits.

**When to run:** after pushing new labels to GitHub, or after syncing new images.
**Skip if:** `REBUILD_DATASET = False` and splits are already current.

In [ ]:
if REBUILD_DATASET:
    cb.rebuild_splits()
else:
    print("Skipped dataset rebuild")

---
## 4. Train Stage 2 — screen / printout / live

In [ ]:
STAGE2_EPOCHS = 40
STAGE2_BATCH = 32
DEVICE = "cuda"

In [ ]:
if TRAIN_STAGE == "stage2":
    !cd /content/id-forensics-model && python scripts/training/train_stage2_screen.py \
        --device {DEVICE} --epochs {STAGE2_EPOCHS} --batch {STAGE2_BATCH}
else:
    print("Skipping Stage 2")

---
## 5. Save weights to Drive

In [ ]:
if TRAIN_STAGE == "stage2":
    cb.save_weights("stage2")
elif TRAIN_STAGE == "stage1":
    cb.save_weights("stage1")
elif TRAIN_STAGE == "stage4":
    cb.save_weights("stage4")
else:
    print("No weights to save")

---
## 6. Evaluate (saved to Drive)

Outputs: `Drive/id-forensics/eval/<stage>/<timestamp>/`
- `report.txt`, `viz/wrong/`, `confusion_matrix.png`

In [ ]:
if RUN_EVAL and TRAIN_STAGE:
    eval_dir = cb.run_eval(TRAIN_STAGE, split=EVAL_SPLIT)
    print(f"\nReview wrong predictions:\n  {eval_dir / 'viz' / 'wrong'}")
else:
    print("Skipping eval")

---
## Optional: Stage 1 (corner detection)

Set `TRAIN_STAGE = "stage1"` in Config.

In [ ]:
# cb.clear_corner_caches()
# !cd /content/id-forensics-model && python scripts/training/train_stage1_corners.py \
#     --model yolov8s-pose.pt --device 0 --epochs 100 --batch 16

## Optional: Stage 4 (ID type classifier)

Set `TRAIN_STAGE = "stage4"` in Config. Requires Stage 1 weights.

In [ ]:
# !cd /content/id-forensics-model && python scripts/deskew_id_type_images.py
# !cd /content/id-forensics-model && python scripts/split_id_type_dataset.py --source all_deskewed
# !cd /content/id-forensics-model && python scripts/training/train_stage4_id_type.py \
#     --device cuda --epochs 40 --batch 32